# TrafficGuard · FastAPI Backend
**COMP47250 · Team Software Project · Project P14 · UCD Summer 2026**

*Soham & Yashi — Backend notebook that loads the trained ResNet18, exposes REST endpoints for the React dashboard, and runs FGSM attacks + Spatial Smoothing defence via ART.*

---
### How this notebook fits into the pipeline

```
mio_tcd_pipeline_v2.ipynb  →  labelled_manifest.csv
trafficguard_model_v1.ipynb →  checkpoints/best.pt
                                      ↓
              trafficguard_backend.ipynb  ←  YOU ARE HERE
                ├── /model/info          (GET)
                ├── /model/metrics       (GET)
                ├── /attack/fgsm         (POST)  ← Payal's logic
                ├── /defence/status      (GET)
                ├── /defence/toggle      (POST)  ← Rian's logic
                ├── /defence/epsilon-sweep (GET)
                └── /demo/full-pipeline  (POST)  ← the money endpoint for the demo
                                      ↓
                         React dashboard (src/)
```

> **Before running this notebook**, make sure:
> 1. `checkpoints/best.pt` exists — Soham's training must have completed at least one epoch.
> 2. All pip installs in Section 0 have been run.
> 3. The path constants in **Section 1** are updated to match your machine.


---
## 0. Install Dependencies

Run this cell once. Installs FastAPI, Uvicorn (the server), ART (for FGSM and Spatial Smoothing), and nest_asyncio which lets Uvicorn run inside a Jupyter notebook.

In [ ]:
# ── One-time installs ─────────────────────────────────────────────────────────
# Run this cell, then restart the kernel before proceeding.

!pip install fastapi uvicorn[standard] python-multipart nest_asyncio --quiet
!pip install adversarial-robustness-toolbox torch torchvision --quiet
!pip install pillow numpy pandas scikit-learn --quiet

print("All dependencies installed.")
print(">>> RESTART THE KERNEL now, then run from Section 1 downward. <<<")


---
## 1. Imports & Configuration

### 1.1 Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import io
import os
import base64
import json
import time
import logging
from pathlib import Path

# ── Async & server ────────────────────────────────────────────────────────────
import asyncio
import nest_asyncio        # allows Uvicorn to run inside Jupyter
import threading
import uvicorn

# ── FastAPI ───────────────────────────────────────────────────────────────────
from fastapi import FastAPI, File, UploadFile, HTTPException, Body
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel

# ── ML & vision ───────────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image

# ── ART — attack & defence ────────────────────────────────────────────────────
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.preprocessor import SpatialSmoothing

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
print("All imports OK.")


### 1.2 Path Configuration

> **Edit only this cell.** Update `CHECKPOINT_PATH` to wherever `best.pt` lives on this machine. Everything else is derived automatically.

In [ ]:
# ── Paths — EDIT THESE FOR YOUR MACHINE ──────────────────────────────────────

# Path to Soham's trained checkpoint (output of trafficguard_model_v1.ipynb)
# Example: r"C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\checkpoints\best.pt"
CHECKPOINT_PATH = Path(r"C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\checkpoints\best.pt")

# Port the FastAPI server will listen on
# The React frontend (src/api/client.js) already points at http://localhost:8000
SERVER_PORT = 8000

# ── Derived constants — no need to edit below ────────────────────────────────
CLASS_NAMES = ['Low', 'Medium', 'High']
LABEL_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"Checkpoint path : {CHECKPOINT_PATH}")
print(f"Checkpoint exists : {CHECKPOINT_PATH.exists()}")
print(f"Server will start at : http://localhost:{SERVER_PORT}")
print(f"Classes : {CLASS_NAMES}")


---
## 2. Model Loader

### 2.1 Architecture

This function **exactly** mirrors the architecture Soham defined in `trafficguard_model_v1.ipynb` Section 3.
It must match byte-for-byte or `load_state_dict` will throw a key mismatch error.

Key points:
- `weights=None` — we supply our own trained weights via `load_state_dict`
- Final FC replaced with `Dropout(0.3) → Linear(512 → 3)` — same as training notebook


In [ ]:
def build_model(num_classes: int = 3) -> nn.Module:
    """
    Rebuild the TrafficGuard ResNet18 architecture.

    Must exactly match Section 3 of trafficguard_model_v1.ipynb:
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes),
        )

    Args:
        num_classes : number of output classes (always 3 for TrafficGuard)

    Returns:
        nn.Module — untrained model skeleton (weights loaded separately)
    """
    model = models.resnet18(weights=None)          # no ImageNet preload
    in_features = model.fc.in_features             # 512 for ResNet18

    # Replace the classifier head to match the training notebook exactly
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


print("build_model() defined.")
print(f"  Final FC: Dropout(0.3) → Linear(512 → {3})")


### 2.2 Checkpoint Loader

Loads `best.pt` once and caches it for the lifetime of the server. Calling `get_model()` a second time returns the already-loaded instance — no disk read, no reallocation.

In [ ]:
# ── Module-level cache — populated by load_checkpoint() below ─────────────────
_model_cache     = None   # the nn.Module in eval() mode
_checkpoint_meta = {}     # epoch, val_acc, class_names read from .pt file


def load_checkpoint(checkpoint_path: Path, device: torch.device) -> dict:
    """
    Load Soham's best.pt checkpoint.

    The checkpoint dict saved by trafficguard_model_v1.ipynb contains:
        {
          'epoch':                int,
          'model_state_dict':     OrderedDict,
          'optimizer_state_dict': OrderedDict,
          'val_acc':              float,
          'class_names':          list[str],
          'label_map':            dict,
        }

    Args:
        checkpoint_path : Path to best.pt
        device          : torch.device (cuda or cpu)

    Returns:
        dict with 'model', 'epoch', 'val_acc', 'class_names'
    """
    global _model_cache, _checkpoint_meta

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found at {checkpoint_path}. "
            "Make sure Soham's training has completed and best.pt is committed to the repo."
        )

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model = build_model(num_classes=3).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()   # IMPORTANT: inference mode — disables Dropout

    _model_cache = model
    _checkpoint_meta = {
        'epoch':       ckpt.get('epoch', '?'),
        'val_acc':     ckpt.get('val_acc', 0.0),
        'class_names': ckpt.get('class_names', CLASS_NAMES),
        'label_map':   ckpt.get('label_map', LABEL_MAP),
    }

    print(f"Checkpoint loaded : epoch {_checkpoint_meta['epoch']}, "
          f"val_acc = {_checkpoint_meta['val_acc']:.2%}")
    print(f"Model on device   : {next(model.parameters()).device}")
    return {'model': model, **_checkpoint_meta}


def get_model() -> nn.Module:
    """
    Return the cached model. Raises RuntimeError if not yet loaded.
    Call load_checkpoint() first (done automatically when the FastAPI app starts).
    """
    if _model_cache is None:
        raise RuntimeError("Model not loaded. Server startup may have failed.")
    return _model_cache


# ── Test load immediately ─────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

try:
    result = load_checkpoint(CHECKPOINT_PATH, DEVICE)
    print(f"✓ Model ready   — {sum(p.numel() for p in result['model'].parameters()):,} parameters")
except FileNotFoundError as e:
    print(f"⚠  {e}")
    print("   The server will still start but /model/info will return loaded=false.")
    print("   Re-run this cell once best.pt exists.")


---
## 3. Inference Helpers

### 3.1 Image Preprocessing

The transform pipeline here is identical to the *val/test* transform in `trafficguard_model_v1.ipynb` Section 2.1 — no augmentation, just resize → tensor → ImageNet normalise.

> **Why no augmentation here?**  
> Augmentation introduces randomness. Inference must be deterministic — same image always produces same prediction.


In [ ]:
# ── Inference transform — matches val/test transform from model notebook ───────
INFERENCE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def pil_to_tensor(image: Image.Image) -> torch.Tensor:
    """
    Convert a PIL image to a normalised [1, 3, 224, 224] tensor ready for the model.

    Args:
        image : PIL.Image in RGB mode

    Returns:
        torch.Tensor shape (1, 3, 224, 224), values normalised with ImageNet stats
    """
    if image.mode != 'RGB':
        image = image.convert('RGB')
    return INFERENCE_TRANSFORM(image).unsqueeze(0)   # add batch dim


def tensor_to_numpy_0_1(tensor: torch.Tensor) -> np.ndarray:
    """
    Convert a (1, 3, 224, 224) normalised tensor back to (1, 3, 224, 224) float32
    in [0, 1] range — the format ART expects.

    We reverse ImageNet normalisation because ART operates in raw pixel space.

    Args:
        tensor : shape (1, 3, 224, 224) normalised tensor

    Returns:
        np.ndarray shape (1, 3, 224, 224) dtype float32, values in [0.0, 1.0]
    """
    mean = torch.tensor(IMAGENET_MEAN, dtype=torch.float32).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  dtype=torch.float32).view(3, 1, 1)
    # Reverse: pixel = tensor * std + mean
    unnorm = (tensor.squeeze(0).cpu() * std + mean).clamp(0, 1)
    return unnorm.unsqueeze(0).numpy()   # shape (1, 3, 224, 224)


def numpy_to_b64_jpeg(arr: np.ndarray) -> str:
    """
    Convert a (1, 3, 224, 224) float32 [0,1] numpy array to a base64 JPEG string.
    Used to send images back to the React frontend for display.

    Args:
        arr : numpy array shape (1, 3, 224, 224), dtype float32, values in [0, 1]

    Returns:
        str — base64-encoded JPEG (no data: prefix — frontend adds that)
    """
    img_hwc = (arr[0].transpose(1, 2, 0) * 255).astype(np.uint8)  # CHW → HWC
    pil_img = Image.fromarray(img_hwc, mode='RGB')
    buffer  = io.BytesIO()
    pil_img.save(buffer, format='JPEG', quality=85)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


print("Preprocessing helpers defined:")
print("  pil_to_tensor()       — PIL → normalised (1,3,224,224) tensor")
print("  tensor_to_numpy_0_1() — tensor → [0,1] numpy for ART")
print("  numpy_to_b64_jpeg()   — numpy → base64 JPEG for frontend")


### 3.2 Clean Prediction

In [ ]:
def predict_pil(image: Image.Image) -> dict:
    """
    Run clean (no attack) inference on a PIL image.

    Args:
        image : PIL.Image.Image (any mode — converted to RGB internally)

    Returns:
        {
          'label':       str   — 'Low' | 'Medium' | 'High',
          'confidence':  float — probability of the predicted class (0–1),
          'probs':       dict  — {'Low': float, 'Medium': float, 'High': float}
        }
    """
    model = get_model()

    tensor = pil_to_tensor(image).to(DEVICE)

    with torch.no_grad():
        logits = model(tensor)                        # shape (1, 3)
        probs  = F.softmax(logits, dim=1).squeeze()   # shape (3,)

    idx        = int(probs.argmax().item())
    label      = IDX_TO_LABEL[idx]
    confidence = float(probs[idx].item())

    return {
        'label':      label,
        'confidence': round(confidence, 4),
        'probs': {
            'Low':    round(float(probs[0].item()), 4),
            'Medium': round(float(probs[1].item()), 4),
            'High':   round(float(probs[2].item()), 4),
        }
    }


print("predict_pil() defined — runs clean ResNet18 inference on a PIL image.")


---
## 4. FGSM Attack (Payal's Logic)

### 4.1 ART Classifier Wrapper

ART needs a `PyTorchClassifier` wrapper around the model. This wrapper handles:
- Automatic ImageNet de-normalisation/re-normalisation around the attack (via `preprocessing`)
- Gradient computation for FGSM
- GPU/CPU dispatch

> **Why `preprocessing=(MEAN, STD)` in the ART classifier?**  
> ART attacks operate in pixel space `[0, 1]`. When it calls `model.forward()`, the model expects *normalised* tensors. Setting `preprocessing` tells ART to normalise inputs before passing them to the model, and to account for this when computing gradients. This matches exactly how Payal wrapped it in `fgsm_attack.ipynb`.


In [ ]:
def build_art_classifier() -> PyTorchClassifier:
    """
    Wrap the loaded ResNet18 in an ART PyTorchClassifier for attack generation.

    This replicates Payal's ART wrapping from fgsm_attack.ipynb (Cell 13),
    updated to use the real trained model instead of her throwaway one.

    Returns:
        PyTorchClassifier ready for .generate() and .predict()
    """
    model = get_model()

    classifier = PyTorchClassifier(
        model=model,
        loss=nn.CrossEntropyLoss(),
        # preprocessing tells ART to apply ImageNet normalisation before
        # feeding to the model (same as Rian's smoothing notebook Cell 8)
        preprocessing=(IMAGENET_MEAN, IMAGENET_STD),
        input_shape=(3, 224, 224),
        nb_classes=3,
        clip_values=(0.0, 1.0),                                # pixel range for ART
        device_type='gpu' if torch.cuda.is_available() else 'cpu',
    )
    return classifier


# Build the classifier once at module level — reused by all attack calls
_art_classifier = None

def get_art_classifier() -> PyTorchClassifier:
    """Return cached ART classifier, building it the first time it's needed."""
    global _art_classifier
    if _art_classifier is None:
        _art_classifier = build_art_classifier()
        print("ART classifier built and cached.")
    return _art_classifier


# Test it now to surface any errors before starting the server
try:
    clf = get_art_classifier()
    print(f"✓ ART classifier ready — input_shape={clf.input_shape}, nb_classes={clf.nb_classes}")
except RuntimeError as e:
    print(f"⚠  Cannot build ART classifier yet: {e}")
    print("   Re-run once the checkpoint is loaded in Section 2.")


### 4.2 FGSM Attack Function

This is Payal's FGSM logic extracted from `fgsm_attack.ipynb` Cell 13, updated to use the real trained model and accept a PIL image input instead of a pre-loaded numpy batch.

In [ ]:
def run_fgsm(image: Image.Image, epsilon: float = 0.1) -> dict:
    """
    Run FGSM on a single PIL image and return predictions + perturbed image.

    Extracted from Payal's fgsm_attack.ipynb (Cell 13).
    Updated to:
      1. Use the real trained model (best.pt) instead of Payal's throwaway model
      2. Accept a PIL image directly (the FastAPI endpoint receives uploaded files)
      3. Return base64 image strings for the React frontend's FrameComparison panel

    Args:
        image   : PIL.Image.Image — the uploaded traffic image
        epsilon : float — perturbation magnitude (0.01 to 0.50)
                  Typical values: 0.05 (subtle), 0.10 (moderate), 0.20 (strong)

    Returns:
        {
          'clean_pred':   str   — 'Low' | 'Medium' | 'High'  (original prediction)
          'attack_pred':  str   — prediction after FGSM perturbation
          'clean_conf':   float — confidence on clean image
          'attack_conf':  float — confidence on adversarial image
          'asr':          int   — 1 if prediction flipped, 0 if not (for this image)
          'epsilon':      float — epsilon used
          'clean_probs':  dict  — full probability dict for clean image
          'attack_probs': dict  — full probability dict for adversarial image
          'clean_image':  str   — base64 JPEG of clean image (for frontend display)
          'attack_image': str   — base64 JPEG of perturbed image (for frontend display)
        }
    """
    classifier = get_art_classifier()

    # Convert PIL image to numpy [0,1] array in ART's expected format (N, C, H, W)
    x_clean = tensor_to_numpy_0_1(pil_to_tensor(image))  # shape (1, 3, 224, 224)

    # ── Step 1: Clean prediction ─────────────────────────────────────────────
    clean_logits = classifier.predict(x_clean)            # shape (1, 3)
    clean_idx    = int(np.argmax(clean_logits, axis=1)[0])
    clean_probs  = dict(zip(CLASS_NAMES,
                            [round(float(v), 4) for v in
                             np.exp(clean_logits[0]) / np.exp(clean_logits[0]).sum()]))
    clean_conf   = float(np.max(clean_logits, axis=1)[0])

    # Note: ART's .predict() returns raw logits, not softmax probs.
    # We normalise manually for the confidence display.
    def logits_to_probs(logits_1d):
        e = np.exp(logits_1d - logits_1d.max())  # numerically stable softmax
        return e / e.sum()

    clean_prob_arr  = logits_to_probs(clean_logits[0])
    clean_conf      = round(float(clean_prob_arr[clean_idx]), 4)
    clean_probs_out = dict(zip(CLASS_NAMES, [round(float(p), 4) for p in clean_prob_arr]))

    # ── Step 2: Generate adversarial example via FGSM ───────────────────────
    # FastGradientMethod: single-step attack in the direction of the loss gradient
    # eps is in pixel space [0, 1] — 0.10 means max 10% pixel shift per channel
    attack = FastGradientMethod(estimator=classifier, eps=epsilon)
    x_adv  = attack.generate(x=x_clean)   # shape (1, 3, 224, 224), values in [0,1]

    # ── Step 3: Prediction on adversarial example ────────────────────────────
    adv_logits   = classifier.predict(x_adv)
    adv_idx      = int(np.argmax(adv_logits, axis=1)[0])
    adv_prob_arr = logits_to_probs(adv_logits[0])
    adv_conf     = round(float(adv_prob_arr[adv_idx]), 4)
    adv_probs_out = dict(zip(CLASS_NAMES, [round(float(p), 4) for p in adv_prob_arr]))

    # ── Step 4: Encode both images for the frontend ──────────────────────────
    clean_b64 = numpy_to_b64_jpeg(x_clean)
    adv_b64   = numpy_to_b64_jpeg(x_adv)

    return {
        'clean_pred':   IDX_TO_LABEL[clean_idx],
        'attack_pred':  IDX_TO_LABEL[adv_idx],
        'clean_conf':   clean_conf,
        'attack_conf':  adv_conf,
        'asr':          int(clean_idx != adv_idx),   # 1 if prediction flipped
        'epsilon':      epsilon,
        'clean_probs':  clean_probs_out,
        'attack_probs': adv_probs_out,
        'clean_image':  clean_b64,
        'attack_image': adv_b64,
    }


print("run_fgsm() defined — FGSM attack on a PIL image.")
print("  Input  : PIL image + epsilon")
print("  Output : clean pred, attack pred, confs, base64 images, ASR flag")


---
## 5. Spatial Smoothing Defence (Rian's Logic)

### 5.1 Defence Setup

This replicates Rian's `spatial_smoothing.ipynb` Cells 4, 6, and 8 — but corrects the one issue in his notebook: the model is now loaded from `best.pt` instead of running as random (uninitialised) weights.

**What Spatial Smoothing does:**  
Before inference, every image is passed through a local averaging filter (3×3 window). Adversarial perturbations are high-frequency noise — tiny pixel differences tuned precisely to fool the model. Averaging nearby pixels destroys this high-frequency signal while keeping the image visually intact. This breaks the perturbation without retraining the model.


In [ ]:
def build_defended_classifier() -> PyTorchClassifier:
    """
    Wrap the trained ResNet18 in an ART classifier with SpatialSmoothing defence.

    Replicates Rian's spatial_smoothing.ipynb Cells 4 + 6 + 8.
    The only change: model weights are loaded from best.pt instead of random init.

    The SpatialSmoothing preprocessor is applied BEFORE inference — ART handles
    this automatically when preprocessing_defences is set on the classifier.

    Returns:
        PyTorchClassifier with SpatialSmoothing(window_size=3) attached
    """
    model = get_model()

    # Exactly as in Rian's notebook Cell 6
    smoothing = SpatialSmoothing(window_size=3, channels_first=True)

    # Exactly as in Rian's notebook Cell 8, but with the real model
    classifier = PyTorchClassifier(
        model=model,
        loss=nn.CrossEntropyLoss(),
        preprocessing=(IMAGENET_MEAN, IMAGENET_STD),    # same as Rian's MEAN/STD
        preprocessing_defences=[smoothing],             # defence applied first
        input_shape=(3, 224, 224),
        nb_classes=3,
        clip_values=(0.0, 1.0),
        device_type='gpu' if torch.cuda.is_available() else 'cpu',
    )
    return classifier


_defended_classifier = None

def get_defended_classifier() -> PyTorchClassifier:
    """Return cached defended classifier (built once, reused)."""
    global _defended_classifier
    if _defended_classifier is None:
        _defended_classifier = build_defended_classifier()
        print("Defended classifier built and cached (SpatialSmoothing window_size=3).")
    return _defended_classifier


try:
    def_clf = get_defended_classifier()
    print(f"✓ Defended classifier ready — preprocessing_defences: {def_clf.preprocessing_defences}")
except RuntimeError as e:
    print(f"⚠  {e}")


### 5.2 Defended Prediction Function

In [ ]:
def predict_with_defence(x_adv: np.ndarray) -> dict:
    """
    Run inference on an adversarial example using the Spatial Smoothing defended model.

    ART automatically applies SpatialSmoothing before the model sees the image.

    Args:
        x_adv : np.ndarray shape (1, 3, 224, 224), dtype float32, values in [0, 1]
                This is the adversarial image produced by run_fgsm()

    Returns:
        {
          'defended_pred'  : str   — prediction after smoothing is applied
          'defended_conf'  : float — confidence of defended prediction
          'defended_probs' : dict  — full probability dict after defence
        }
    """
    def_clf = get_defended_classifier()

    def logits_to_probs(logits_1d):
        e = np.exp(logits_1d - logits_1d.max())
        return e / e.sum()

    defended_logits  = def_clf.predict(x_adv)          # smoothing applied internally
    def_idx          = int(np.argmax(defended_logits, axis=1)[0])
    def_prob_arr     = logits_to_probs(defended_logits[0])
    def_conf         = round(float(def_prob_arr[def_idx]), 4)
    def_probs_out    = dict(zip(CLASS_NAMES, [round(float(p), 4) for p in def_prob_arr]))

    return {
        'defended_pred':   IDX_TO_LABEL[def_idx],
        'defended_conf':   def_conf,
        'defended_probs':  def_probs_out,
    }


print("predict_with_defence() defined.")
print("  Input  : adversarial numpy array (from run_fgsm)")
print("  Output : defended prediction + confidence")


---
## 6. Epsilon Sweep Helper

The React dashboard's `EpsilonChart` panel expects a list of `{epsilon, asr, robust_acc}` data points from the `/defence/epsilon-sweep` endpoint. This function pre-computes those on a small reference set so the endpoint can return them instantly.


In [ ]:
# ── Pre-computed epsilon sweep results (populated by sweep_epsilon_range) ──────
_epsilon_sweep_cache = None


def sweep_epsilon_range(
    sample_images: list,
    epsilons: list = None,
) -> list:
    """
    Run FGSM at multiple epsilon values on a list of PIL images and compute
    ASR and Robust Accuracy at each epsilon level.

    Used to populate the EpsilonChart in the React dashboard.

    Args:
        sample_images : list of PIL.Image.Image — test images to attack
        epsilons      : list of float — epsilon values to try
                        default: [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

    Returns:
        list of dicts:
          [{'epsilon': 0.01, 'asr': 0.12, 'robust_acc': 0.88}, ...]
    """
    global _epsilon_sweep_cache

    if epsilons is None:
        epsilons = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

    classifier = get_art_classifier()
    results = []

    # Convert all sample images to numpy once (avoid repeated PIL→numpy conversion)
    x_batch  = np.concatenate([
        tensor_to_numpy_0_1(pil_to_tensor(img)) for img in sample_images
    ], axis=0)   # shape (N, 3, 224, 224)

    # Clean predictions (ground reference for ASR computation)
    clean_logits = classifier.predict(x_batch)
    clean_preds  = np.argmax(clean_logits, axis=1)

    for eps in epsilons:
        attack    = FastGradientMethod(estimator=classifier, eps=eps)
        x_adv     = attack.generate(x=x_batch)
        adv_preds = np.argmax(classifier.predict(x_adv), axis=1)

        asr        = float(np.mean(adv_preds != clean_preds))
        robust_acc = float(np.mean(adv_preds == clean_preds))  # fraction still correct

        results.append({
            'epsilon':    round(eps, 3),
            'asr':        round(asr * 100, 2),         # as percentage
            'robust_acc': round(robust_acc * 100, 2),  # as percentage
        })

    _epsilon_sweep_cache = results
    return results


print("sweep_epsilon_range() defined.")
print("  Call this once with a batch of test images to pre-compute the epsilon chart data.")
print("  Results cached in _epsilon_sweep_cache and served by /defence/epsilon-sweep.")


---
## 7. Request / Response Schemas

FastAPI uses Pydantic models for automatic validation and documentation. These schemas match exactly what `src/api/client.js` sends and receives.


In [ ]:
# ── Request bodies ─────────────────────────────────────────────────────────────

class FGSMRequest(BaseModel):
    """
    Body for POST /attack/fgsm
    Sent by: src/api/client.js → runFGSM(imageB64, epsilon)
    """
    image:   str    # base64-encoded image (no data: prefix)
    epsilon: float  # perturbation magnitude, e.g. 0.10


class DefenceToggleRequest(BaseModel):
    """
    Body for POST /defence/toggle
    Sent by: src/context/AttackContext.jsx → toggleDef()
    """
    name:    str   # 'smooth' | 'advtrain' | 'jpeg' | 'rs' | 'ensemble'
    enabled: bool


# ── Response shape helpers ─────────────────────────────────────────────────────

def make_error(message: str, status_code: int = 400) -> JSONResponse:
    """Standard error response — keeps the frontend's error handling consistent."""
    return JSONResponse(
        status_code=status_code,
        content={'error': message}
    )


print("Pydantic schemas defined:")
print("  FGSMRequest        — image (base64) + epsilon")
print("  DefenceToggleRequest — name + enabled")


---
## 8. FastAPI Application

### 8.1 App Initialisation & CORS

CORS (Cross-Origin Resource Sharing) must be enabled so the React dev server on `localhost:5173` can call the API on `localhost:8000`. Without this, the browser blocks all requests.


In [ ]:
app = FastAPI(
    title        = "TrafficGuard Security API",
    description  = "Adversarial attack and defence evaluation for TrafficGuard congestion classifier",
    version      = "1.0.0",
)

# ── CORS — allow the React frontend (Vite dev server) to call this API ─────────
app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["http://localhost:5173", "http://localhost:3000", "*"],
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)

# ── Startup: load model once when the server boots ────────────────────────────
@app.on_event("startup")
async def startup_event():
    """
    Load the trained checkpoint and warm up the ART classifiers
    once when the server starts — not on every request.
    """
    global _model_cache, _art_classifier, _defended_classifier

    print("[startup] Loading checkpoint...")
    try:
        load_checkpoint(CHECKPOINT_PATH, DEVICE)
        print("[startup] Building ART classifiers...")
        get_art_classifier()
        get_defended_classifier()
        print("[startup] ✓ All components ready.")
    except FileNotFoundError as e:
        print(f"[startup] ⚠  {e}")
        print("[startup]    Endpoints will return 503 until checkpoint is available.")


print("FastAPI app created.")
print("  CORS: allow_origins=['http://localhost:5173', 'http://localhost:3000', '*']")


### 8.2 Model Endpoints

These serve the static model metadata that the dashboard's `StatCard` and `Model` page display.

In [ ]:
@app.get("/model/info")
async def model_info():
    """
    GET /model/info
    Called by: src/api/client.js → getModelInfo()
    Used by: Dashboard StatCard, Model page

    Returns model metadata if checkpoint is loaded, graceful error if not.
    """
    if _model_cache is None:
        return JSONResponse(
            status_code=503,
            content={
                'loaded': False,
                'error':  'Checkpoint not found — training may still be in progress.'
            }
        )

    total_params = sum(p.numel() for p in _model_cache.parameters())

    return {
        'loaded':        True,
        'architecture':  'ResNet18',
        'num_classes':   3,
        'class_names':   CLASS_NAMES,
        'total_params':  total_params,
        'checkpoint': {
            'epoch':   _checkpoint_meta.get('epoch'),
            'val_acc': round(_checkpoint_meta.get('val_acc', 0.0) * 100, 2),
        },
        'device': str(DEVICE),
    }


@app.get("/model/metrics")
async def model_metrics():
    """
    GET /model/metrics
    Called by: src/api/client.js → getMetrics()
    Used by: Dashboard StatCard row (cleanAcc, robustAcc, asr, certifiedRadius)

    Returns the training metrics from best.pt.
    Robust accuracy and ASR are initially placeholders —
    they update dynamically as attacks are run via /attack/fgsm.
    """
    if _model_cache is None:
        return JSONResponse(status_code=503, content={'error': 'Model not loaded'})

    val_acc_pct = round(_checkpoint_meta.get('val_acc', 0.0) * 100, 2)

    return {
        'cleanAcc':       val_acc_pct,
        'robustAcc':      None,   # populated after first /attack/fgsm call
        'asr':            None,   # populated after first /attack/fgsm call
        'certifiedRadius': 0.25,  # Randomized Smoothing placeholder (Rian's extended work)
        'epoch':          _checkpoint_meta.get('epoch'),
    }


print("Model endpoints registered:")
print("  GET /model/info    — architecture, params, checkpoint epoch/val_acc")
print("  GET /model/metrics — cleanAcc, robustAcc, asr (placeholders until attack runs)")


### 8.3 Attack Endpoints

In [ ]:
@app.post("/attack/fgsm")
async def fgsm_endpoint(body: FGSMRequest):
    """
    POST /attack/fgsm
    Called by: src/api/client.js → runFGSM(imageB64, epsilon)
    Used by: AttackLab.jsx 'Run Attack' button → Dashboard FrameComparison panel

    Accepts a base64-encoded image and epsilon value.
    Returns clean prediction, adversarial prediction, and both images as base64.

    The response shape is designed to match what StreamContext expects for
    a 'frame' object (latestFrame in FrameComparison.jsx):
        clean_pred, attack_pred, clean_conf, attack_conf,
        clean_image (base64), attack_image (base64)
    """
    if _model_cache is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    # Validate epsilon range
    if not (0.001 <= body.epsilon <= 0.5):
        raise HTTPException(
            status_code=422,
            detail=f"epsilon must be between 0.001 and 0.5, got {body.epsilon}"
        )

    try:
        # Decode base64 image → PIL
        image_bytes = base64.b64decode(body.image)
        image       = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image data: {e}")

    try:
        result = run_fgsm(image, epsilon=body.epsilon)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Attack failed: {e}")

    # Add metadata the React StreamContext expects on a 'frame' object
    result.update({
        'type':        'frame',          # StreamContext.handler checks data.type === 'frame'
        'frame_id':    str(int(time.time() * 1000) % 100000),
        'timestamp':   time.strftime('%H:%M:%S'),
        'attack_type': 'FGSM',
    })

    return result


@app.post("/attack/pgd")
async def pgd_endpoint():
    """
    POST /attack/pgd
    NOT IMPLEMENTED for interim demo — PGD is assigned to the final submission.
    Returns 501 so the frontend gets a clean error instead of crashing.
    """
    return JSONResponse(
        status_code=501,
        content={'error': 'PGD attack not implemented yet — planned for final submission.'}
    )


@app.post("/attack/poison/labelflip")
async def labelflip_endpoint():
    """POST /attack/poison/labelflip — stub, not implemented for interim demo."""
    return JSONResponse(
        status_code=501,
        content={'error': 'Label flipping attack not implemented for interim demo.'}
    )


@app.post("/attack/poison/backdoor")
async def backdoor_endpoint():
    """POST /attack/poison/backdoor — stub, not implemented for interim demo."""
    return JSONResponse(
        status_code=501,
        content={'error': 'Backdoor attack not implemented for interim demo.'}
    )


print("Attack endpoints registered:")
print("  POST /attack/fgsm          — LIVE (Payal's logic)")
print("  POST /attack/pgd           — 501 stub")
print("  POST /attack/poison/*      — 501 stubs")


### 8.4 Defence Endpoints

In [ ]:
# ── In-memory defence state (toggled by the frontend) ─────────────────────────
_defence_state = {
    'smooth':    {'enabled': True,  'window_size': 3},
    'advtrain':  {'enabled': False, 'note': 'Adversarial training — full retraining required'},
    'jpeg':      {'enabled': False, 'quality': 75},
    'rs':        {'enabled': False, 'sigma': 0.25},
    'ensemble':  {'enabled': False},
}


@app.get("/defence/status")
async def defence_status():
    """
    GET /defence/status
    Called by: src/api/client.js → getDefenceStatus()
    Used by: Defences page, DefenceStatus panel

    Returns current enabled/disabled state of all defences.
    """
    return _defence_state


@app.post("/defence/toggle")
async def defence_toggle(body: DefenceToggleRequest):
    """
    POST /defence/toggle
    Called by: src/api/client.js → toggleDefence(name, enabled)
    Triggered by: src/context/AttackContext.jsx → toggleDef()

    Updates the in-memory defence state.
    For spatial smoothing ('smooth'), also rebuilds the ART defended classifier
    so subsequent calls to /demo/full-pipeline use the updated state.
    """
    global _defended_classifier

    if body.name not in _defence_state:
        raise HTTPException(
            status_code=404,
            detail=f"Unknown defence '{body.name}'. "
                   f"Valid names: {list(_defence_state.keys())}"
        )

    _defence_state[body.name]['enabled'] = body.enabled

    # If spatial smoothing was toggled, invalidate the cached defended classifier
    # so it gets rebuilt with the new state on the next request
    if body.name == 'smooth':
        _defended_classifier = None

    return {
        'name':    body.name,
        'enabled': body.enabled,
        'state':   _defence_state,
    }


@app.get("/defence/epsilon-sweep")
async def epsilon_sweep_endpoint(attack: str = 'fgsm'):
    """
    GET /defence/epsilon-sweep?attack=fgsm
    Called by: src/api/client.js → getEpsilonSweep(attackType)
    Used by: EpsilonChart panel in the dashboard

    Returns pre-computed ASR at each epsilon value.
    If no sweep has been run yet, returns a static placeholder curve.
    """
    if _epsilon_sweep_cache is not None:
        return {'attack': attack, 'data': _epsilon_sweep_cache}

    # Placeholder data shown before the sweep is run
    # Shape matches what EpsilonChart expects: [{epsilon, asr, robust_acc}]
    placeholder = [
        {'epsilon': 0.01, 'asr': 5.0,  'robust_acc': 95.0},
        {'epsilon': 0.05, 'asr': 18.0, 'robust_acc': 82.0},
        {'epsilon': 0.10, 'asr': 38.0, 'robust_acc': 62.0},
        {'epsilon': 0.20, 'asr': 61.0, 'robust_acc': 39.0},
        {'epsilon': 0.30, 'asr': 78.0, 'robust_acc': 22.0},
        {'epsilon': 0.50, 'asr': 91.0, 'robust_acc': 9.0},
    ]
    return {'attack': attack, 'data': placeholder, 'note': 'placeholder — run sweep for real data'}


@app.get("/defence/certified-radius")
async def certified_radius(sigma: float = 0.25):
    """
    GET /defence/certified-radius?sigma=0.25
    Stub endpoint — Randomized Smoothing is Rian's extended work for final submission.
    """
    return JSONResponse(
        status_code=501,
        content={'error': 'Certified radius not implemented for interim demo.'}
    )


print("Defence endpoints registered:")
print("  GET  /defence/status         — current enabled/disabled state")
print("  POST /defence/toggle         — toggle a defence on/off")
print("  GET  /defence/epsilon-sweep  — ASR vs epsilon curve for EpsilonChart")
print("  GET  /defence/certified-radius — 501 stub")


### 8.5 Demo Full-Pipeline Endpoint

This is the **main endpoint for the interim demo**. It runs the complete story in one call: clean prediction → FGSM attack → (if smoothing enabled) defence recovery. The React `AttackLab.jsx` 'Run Attack' button should call this.

In [ ]:
@app.post("/demo/full-pipeline")
async def demo_full_pipeline(file: UploadFile = File(...),
                              epsilon: float = 0.10):
    """
    POST /demo/full-pipeline
    The money endpoint for the interim demo.

    Accepts an image file upload (not base64) for easier browser FormData usage.
    Runs the full story: clean → attack → defence.

    Args:
        file    : uploaded image file (multipart/form-data)
        epsilon : FGSM epsilon (query param or form field), default 0.10

    Returns:
        {
          'clean_pred':     'High',
          'attack_pred':    'Low',       ← prediction after FGSM
          'defended_pred':  'High',      ← prediction after Spatial Smoothing
          'clean_conf':     0.91,
          'attack_conf':    0.85,
          'defended_conf':  0.78,
          'prediction_flipped':  true,   ← did FGSM fool the model?
          'defence_recovered':   true,   ← did Spatial Smoothing fix it?
          'epsilon':        0.10,
          'clean_image':    '<base64>',
          'attack_image':   '<base64>',
          'type':           'frame',      ← StreamContext recognises this
          'frame_id':       '12345',
          'timestamp':      '14:22:01',
          'attack_type':    'FGSM',
        }
    """
    if _model_cache is None:
        raise HTTPException(status_code=503, detail="Model not loaded. Training still in progress?")

    # ── Read uploaded file ─────────────────────────────────────────────────────
    contents = await file.read()
    try:
        image = Image.open(io.BytesIO(contents)).convert('RGB')
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Cannot read image file: {e}")

    # ── Step 1 & 2: Clean prediction + FGSM attack ────────────────────────────
    try:
        attack_result = run_fgsm(image, epsilon=epsilon)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"FGSM attack failed: {e}")

    # ── Step 3: Spatial Smoothing defence (if enabled) ────────────────────────
    defended_result = {}
    if _defence_state.get('smooth', {}).get('enabled', False):
        try:
            # Re-create the adversarial numpy array for the defence step
            x_clean = tensor_to_numpy_0_1(pil_to_tensor(image))
            classifier = get_art_classifier()
            attack_obj = FastGradientMethod(estimator=classifier, eps=epsilon)
            x_adv      = attack_obj.generate(x=x_clean)

            defended_result = predict_with_defence(x_adv)
        except Exception as e:
            defended_result = {'defended_pred': None, 'defended_conf': None,
                               'error': str(e)}
    else:
        defended_result = {
            'defended_pred':   None,
            'defended_conf':   None,
            'note': 'Spatial Smoothing is disabled. Enable it via /defence/toggle.'
        }

    # ── Assemble response ──────────────────────────────────────────────────────
    clean_pred    = attack_result['clean_pred']
    attack_pred   = attack_result['attack_pred']
    defended_pred = defended_result.get('defended_pred')

    return {
        # Core predictions
        'clean_pred':     clean_pred,
        'attack_pred':    attack_pred,
        'defended_pred':  defended_pred,
        # Confidence scores
        'clean_conf':     attack_result['clean_conf'],
        'attack_conf':    attack_result['attack_conf'],
        'defended_conf':  defended_result.get('defended_conf'),
        # Story flags — used by the demo narrative
        'prediction_flipped': clean_pred != attack_pred,
        'defence_recovered':  (defended_pred is not None and defended_pred == clean_pred),
        # Probability breakdowns for the Model page
        'clean_probs':     attack_result['clean_probs'],
        'attack_probs':    attack_result['attack_probs'],
        'defended_probs':  defended_result.get('defended_probs'),
        # Attack config
        'epsilon':         epsilon,
        # StreamContext frame fields
        'type':            'frame',
        'frame_id':        str(int(time.time() * 1000) % 100000),
        'timestamp':       time.strftime('%H:%M:%S'),
        'attack_type':     'FGSM',
        # Images for FrameComparison panel
        'clean_image':     attack_result['clean_image'],
        'attack_image':    attack_result['attack_image'],
    }


print("Demo endpoint registered:")
print("  POST /demo/full-pipeline — clean → FGSM → Spatial Smoothing in one call")


### 8.6 Report & Utility Endpoints

In [ ]:
@app.post("/report/generate")
async def report_generate():
    """
    POST /report/generate
    Stub — PDF report generation is a final submission feature.
    """
    return JSONResponse(
        status_code=501,
        content={'error': 'Report generation not implemented for interim demo.'}
    )


@app.get("/health")
async def health():
    """
    GET /health — sanity check endpoint.
    Useful for checking the server is up before the frontend loads.
    """
    return {
        'status':       'ok',
        'model_loaded': _model_cache is not None,
        'device':       str(DEVICE),
        'timestamp':    time.strftime('%Y-%m-%d %H:%M:%S'),
    }


@app.get("/")
async def root():
    """GET / — redirect to interactive API docs."""
    return {
        'message': 'TrafficGuard Security API',
        'docs':    'http://localhost:8000/docs',
        'health':  'http://localhost:8000/health',
    }


print("Utility endpoints registered:")
print("  GET  /health          — server health check")
print("  POST /report/generate — 501 stub")
print("  GET  /                — API info")


---
## 9. Start the Server

### 9.1 How to run

`nest_asyncio` patches the event loop so Uvicorn can run inside a Jupyter notebook cell. Without it you get a `RuntimeError: This event loop is already running.`

> **After running this cell**, the server stays alive in the background. You can still run other cells below it.
> To stop it: **Kernel → Restart Kernel**.

The React frontend expects the server at `http://localhost:8000` — this is already set in `src/api/client.js`.


In [ ]:
# ── Apply nest_asyncio so Uvicorn runs inside the notebook event loop ─────────
nest_asyncio.apply()

def run_server():
    """Run Uvicorn in a background thread so this cell doesn't block."""
    uvicorn.run(
        app,
        host    = "0.0.0.0",
        port    = SERVER_PORT,
        reload  = False,       # reload=True doesn't work inside notebooks
        log_level = "info",
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give Uvicorn a moment to boot
import time
time.sleep(2)

print(f"✓ Server started at http://localhost:{SERVER_PORT}")
print(f"  Interactive docs : http://localhost:{SERVER_PORT}/docs")
print(f"  Health check     : http://localhost:{SERVER_PORT}/health")
print()
print("The server runs in a background thread.")
print("To stop it: Kernel → Restart Kernel")


### 9.2 Quick Smoke Test

Run this cell to confirm the server is responding correctly to all key endpoints before switching to the React frontend.

In [ ]:
import urllib.request
import json as _json

BASE = f"http://localhost:{SERVER_PORT}"

def check(method, path, label, body=None):
    """Simple HTTP check without needing the requests library."""
    try:
        if body:
            data = _json.dumps(body).encode()
            req  = urllib.request.Request(BASE + path, data=data,
                                          method=method,
                                          headers={'Content-Type': 'application/json'})
        else:
            req = urllib.request.Request(BASE + path, method=method)

        with urllib.request.urlopen(req, timeout=5) as resp:
            result = _json.loads(resp.read())
            print(f"  ✓ {label:<30} → {resp.status} {list(result.keys())[:4]}")
    except urllib.error.HTTPError as e:
        body_text = e.read().decode()[:80]
        print(f"  ~ {label:<30} → {e.code} (expected for stubs): {body_text}")
    except Exception as e:
        print(f"  ✗ {label:<30} → ERROR: {e}")


print("Running smoke tests...")
print()
check("GET",  "/health",          "GET /health")
check("GET",  "/",                "GET /")
check("GET",  "/model/info",      "GET /model/info")
check("GET",  "/model/metrics",   "GET /model/metrics")
check("GET",  "/defence/status",  "GET /defence/status")
check("GET",  "/defence/epsilon-sweep?attack=fgsm", "GET /epsilon-sweep")
check("POST", "/attack/pgd",      "POST /attack/pgd (stub)")
check("POST", "/report/generate", "POST /report/generate (stub)")
print()
print("Smoke tests complete.")
print("For /attack/fgsm and /demo/full-pipeline, use the React frontend or the test below.")


### 9.3 Full Pipeline Test (with a real image)

This tests the main demo endpoint end-to-end. Replace `TEST_IMAGE_PATH` with any `.jpg` from your MIO-TCD test set.

In [ ]:
import base64, io

# ── Replace with any MIO-TCD test image on your machine ─────────────────────
TEST_IMAGE_PATH = Path(r"C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\data\raw\MIO-TCD-Localization\train\00000001.jpg")

if TEST_IMAGE_PATH.exists():
    # Load and encode as base64 for the /attack/fgsm endpoint
    with open(TEST_IMAGE_PATH, 'rb') as f:
        raw_bytes = f.read()
    img_b64 = base64.b64encode(raw_bytes).decode('utf-8')

    # ── Test /attack/fgsm ──────────────────────────────────────────────────────
    req_body = _json.dumps({'image': img_b64, 'epsilon': 0.10}).encode()
    req = urllib.request.Request(
        f"{BASE}/attack/fgsm",
        data=req_body,
        method='POST',
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            result = _json.loads(resp.read())
        print("POST /attack/fgsm result:")
        print(f"  Clean prediction  : {result['clean_pred']} (conf={result['clean_conf']:.0%})")
        print(f"  Attack prediction : {result['attack_pred']} (conf={result['attack_conf']:.0%})")
        flipped = result['clean_pred'] != result['attack_pred']
        print(f"  Prediction flipped: {flipped}")
        print(f"  Clean image size  : {len(result.get('clean_image',''))} chars (base64)")
        print(f"  Attack image size : {len(result.get('attack_image',''))} chars (base64)")
        print()
        if flipped:
            print(f"  ✓ FGSM successfully fooled the model at ε=0.10")
        else:
            print(f"  ~ FGSM did not flip this specific image at ε=0.10")
            print(f"    Try a higher epsilon or a different test image.")
    except Exception as e:
        print(f"  ✗ /attack/fgsm test failed: {e}")
else:
    print(f"Test image not found at: {TEST_IMAGE_PATH}")
    print("Update TEST_IMAGE_PATH to any .jpg from MIO-TCD-Localization/train/")


---
## 10. Required Changes in Other Notebooks

This section documents exactly what Payal and Rian need to change in their existing notebooks so they work with the real trained model. **No code in this notebook changes** — all changes are in their files.

---

### 10.1 Changes in Payal's `fgsm_attack.ipynb`

**Problem:** Payal's notebook trains 4 separate models from scratch (ResNet50, then ResNet18 iterations), each on tiny subsets with her own congestion labels. The FGSM attack in Cell 13 runs against a model trained on 30 images for 2 epochs — not the real TrafficGuard model.

**The fix:** Delete cells 6–13 (everything from the first import block to the end of the training loop) and replace them with the three cells below.

---

**New Cell A — Replace cells 6 through 12 entirely:**
```python
import os, numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod

# ── PATHS — update these to match your machine ────────────────────────────────
MANIFEST_PATH   = r"C:\Users\Dell\Downloads\MIO-TCD-Localization\MIO-TCD-Localization\pipeline_output\labelled_manifest.csv"
IMAGES_DIR      = r"C:\Users\Dell\Downloads\MIO-TCD-Localization\MIO-TCD-Localization\train"
CHECKPOINT_PATH = r"C:\path\to\trafficguard-p14\checkpoints\best.pt"
# ─────────────────────────────────────────────────────────────────────────────

LABEL_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}
CLASS_NAMES = ['Low', 'Medium', 'High']
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load Soham's trained model (same architecture as trafficguard_model_v1.ipynb) ─
def build_model(num_classes=3):
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

model = build_model().to(device)
ckpt  = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded checkpoint: epoch {ckpt.get('epoch','?')}, val_acc={ckpt.get('val_acc',0):.2%}")
```

**New Cell B — Dataset from Walid's manifest (replaces Payal's own congestion_labels.csv):**
```python
class MIOTCDDataset(Dataset):
    def __init__(self, manifest_path, split, image_dir, transform=None):
        df = pd.read_csv(manifest_path, dtype={'image_id': str})
        self.df        = df[df['split'] == split].reset_index(drop=True)
        self.df['idx'] = self.df['congestion_label'].map(LABEL_MAP)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.image_dir, f"{row['image_id']}.jpg")
        image = Image.open(path).convert('RGB')
        label = int(row['idx'])
        if self.transform:
            image = self.transform(image)
        return image, label

transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])
test_ds   = MIOTCDDataset(MANIFEST_PATH, 'test', IMAGES_DIR, transform)
print(f"Test set from Walid's manifest: {len(test_ds)} images")
```

**New Cell C — ART classifier + FGSM attack (keep this, it replaces Cell 13's training loop + attack block):**
```python
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

classifier = PyTorchClassifier(
    model=model,
    loss=nn.CrossEntropyLoss(),
    preprocessing=(IMAGENET_MEAN, IMAGENET_STD),
    input_shape=(3, 224, 224),
    nb_classes=3,
    clip_values=(0.0, 1.0),
    device_type='gpu' if torch.cuda.is_available() else 'cpu',
)

test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
x_test_list, y_test_list = [], []
for images, labels in test_loader:
    x_test_list.append(images.numpy())
    y_test_list.append(labels.numpy())
x_test = np.concatenate(x_test_list)
y_test = np.concatenate(y_test_list)

clean_preds = np.argmax(classifier.predict(x_test), axis=1)
clean_acc   = np.mean(clean_preds == y_test)
print(f"Clean accuracy (real model on real test set): {clean_acc:.2%}")

epsilons = [0.01, 0.05, 0.10, 0.20]
results  = []
for eps in epsilons:
    attack    = FastGradientMethod(estimator=classifier, eps=eps)
    x_adv     = attack.generate(x=x_test)
    adv_preds = np.argmax(classifier.predict(x_adv), axis=1)
    acc       = np.mean(adv_preds == y_test)
    asr       = np.mean(adv_preds != clean_preds)
    results.append({'eps': eps, 'attacked_acc': acc, 'asr': asr})
    print(f"ε={eps} | Accuracy: {acc:.2%} | ASR: {asr:.2%}")

os.makedirs('outputs', exist_ok=True)
pd.DataFrame(results).to_csv('outputs/fgsm_results.csv', index=False)
print("Saved to outputs/fgsm_results.csv")
```

**Cell 14 (the image-saving cell) — keep exactly as-is.** It already references `x_adv`, `x_test`, `y_test` generically — no changes needed.

---

### 10.2 Changes in Rian's `spatial_smoothing.ipynb`

**Problem:** Cell 4 builds the model but never loads `best.pt`. The model runs with random (uninitialised) weights. Smoothing a random model doesn't demonstrate anything meaningful.

**The only change needed — in Cell 4, replace the last 3 lines:**
```python
# ── BEFORE (Cell 4, last 3 lines) ────────────────────────────────────────────
model = build_resnet18_model().to(device)
model.eval()   # Defence is applied during inference

# ── AFTER — add checkpoint loading between build and eval ────────────────────
CHECKPOINT_PATH = r"C:\path\to\trafficguard-p14\checkpoints\best.pt"

model = build_resnet18_model().to(device)
ckpt  = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded checkpoint: epoch {ckpt.get('epoch','?')}, val_acc={ckpt.get('val_acc',0):.2%}")
```

**Everything else in Rian's notebook stays the same.** Cells 6 (SpatialSmoothing init), 8 (ART classifier with preprocessing_defences), and 10 (visualisation) are all correct and don't need changes.

---

### 10.3 Changes in `src/api/client.js` (Frontend — Aryan)

The client already defines `runFGSM` correctly. One addition is needed — a function to call `/demo/full-pipeline` with a file upload, which is more convenient for the `AttackLab.jsx` 'Run Attack' button:

**Add this function to the end of `src/api/client.js`:**
```js
// Full pipeline endpoint — used by AttackLab 'Run Attack' button
export const runFullPipeline = (imageFile, epsilon = 0.10) => {
  const form = new FormData()
  form.append('file', imageFile)
  return api.post(`/demo/full-pipeline?epsilon=${epsilon}`, form, {
    headers: { 'Content-Type': 'multipart/form-data' },
  })
}
```

**And in `AttackLab.jsx`**, the 'Run Attack' button's `onClick` handler should call `runFullPipeline` and pass the result to `StreamContext` as a frame. The `type: 'frame'` field is already in the response — `StreamContext` will pick it up automatically and update `latestFrame`, which `FrameComparison.jsx` reads.

---

### 10.4 Summary of all changes

| File | What changes | Who does it |
|---|---|---|
| `fgsm_attack.ipynb` | Delete cells 6–13, add 3 new cells (model load + Walid's manifest + ART attack) | Payal |
| `spatial_smoothing.ipynb` | Cell 4: add `torch.load` + `load_state_dict` (3 lines) | Rian |
| `src/api/client.js` | Add `runFullPipeline()` at the bottom | Aryan |
| `trafficguard_backend.ipynb` | Run once — starts the server | Soham & Yashi |
| `checkpoints/best.pt` | Commit to repo once training finishes | Soham |


---
## 11. Summary

### Endpoint map

| Endpoint | Method | Status | Used by |
|---|---|---|---|
| `/health` | GET | ✅ Live | Server monitoring |
| `/model/info` | GET | ✅ Live | Dashboard StatCards, Model page |
| `/model/metrics` | GET | ✅ Live | Dashboard StatCards |
| `/attack/fgsm` | POST | ✅ Live | AttackLab, FrameComparison |
| `/demo/full-pipeline` | POST | ✅ Live | AttackLab 'Run Attack' button |
| `/defence/status` | GET | ✅ Live | Defences page, DefenceStatus panel |
| `/defence/toggle` | POST | ✅ Live | DefenceToggle component |
| `/defence/epsilon-sweep` | GET | ✅ Live | EpsilonChart panel |
| `/attack/pgd` | POST | 🔲 501 stub | Final submission |
| `/attack/poison/*` | POST | 🔲 501 stub | Final submission |
| `/defence/certified-radius` | GET | 🔲 501 stub | Final submission |
| `/report/generate` | POST | 🔲 501 stub | Final submission |

### What changes in other notebooks

| File | Change | Lines |
|---|---|---|
| `fgsm_attack.ipynb` | Delete cells 6–13, add 3 new cells | ~80 lines |
| `spatial_smoothing.ipynb` | Cell 4: add checkpoint loading | 3 lines |
| `src/api/client.js` | Add `runFullPipeline()` | 7 lines |

### Run order for the demo

1. Soham runs `trafficguard_model_v1.ipynb` → `checkpoints/best.pt`
2. Soham/Yashi run `trafficguard_backend.ipynb` → server at `localhost:8000`
3. Aryan starts React dev server → `npm run dev` → `localhost:5173`
4. Upload a MIO-TCD image → click 'Run Attack' → FrameComparison shows clean vs adversarial
5. Toggle Spatial Smoothing on → re-run → see prediction recover


In [ ]:
print("TrafficGuard Backend — Summary")
print("=" * 55)
print(f"Server          : http://localhost:{SERVER_PORT}")
print(f"Docs            : http://localhost:{SERVER_PORT}/docs")
print(f"Model loaded    : {_model_cache is not None}")
if _model_cache is not None:
    print(f"Checkpoint epoch: {_checkpoint_meta.get('epoch', '?')}")
    print(f"Val accuracy    : {_checkpoint_meta.get('val_acc', 0):.2%}")
print()
print("Live endpoints  :")
for ep in ['/health', '/model/info', '/model/metrics',
           '/attack/fgsm', '/demo/full-pipeline',
           '/defence/status', '/defence/toggle',
           '/defence/epsilon-sweep']:
    print(f"  {ep}")
